# Local RAG over Shakespeare

Build a small retrieval-augmented generation (RAG) pipeline that retrieves passages from a local Shakespeare corpus and asks a local Ollama model to answer from those passages.

The notebook is designed for learning: each stage is visible, citations are character offsets rather than production document identifiers, and the final out-of-domain question tests whether the model abstains instead of using outside knowledge.

## Goal

This notebook demonstrates five parts of a basic RAG system:

1. load and clean a text corpus;
2. split documents into overlapping chunks;
3. embed and index the chunks with FAISS;
4. retrieve evidence for a question; and
5. generate a grounded answer with a local language model.

## Setup

Run these commands in a terminal from the repository root:

```bash
brew install ollama
brew services start ollama
ollama pull llama3:8b
python -m venv .venv
source .venv/bin/activate
python -m pip install jupyter faiss-cpu sentence-transformers ollama numpy
jupyter lab rag.ipynb
```

Place a UTF-8 text copy of Shakespeare's complete works at `data/shakespeare.txt`. The `data/` directory is intentionally ignored by Git; review the source's license and terms before downloading or redistributing it.

The sentence-transformer model is downloaded on first use. Ollama must be running before the generation cells.

## 1. Load and clean the corpus

In [ ]:
from pathlib import Path
import re

DATA_PATH = Path("data/shakespeare.txt")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/shakespeare.txt. See the Setup section above."
    )

raw = DATA_PATH.read_text(encoding="utf-8", errors="ignore")

# Heuristic: treat an all-caps line after a blank line as a work/section title.
blocks = re.split(r"\n\s*\n(?=[A-Z][A-Z \-']+\n)", raw)
documents = []
for block in blocks:
    lines = [line.strip() for line in block.splitlines() if line.strip()]
    if not lines:
        continue
    title = lines[0] if lines[0].isupper() else "UNKNOWN"
    text = " ".join(lines[1:] if title != "UNKNOWN" else lines)
    if len(text) >= 500:
        documents.append({"work": title.title(), "text": text})

if not documents:
    raise ValueError("No document sections were detected; inspect the source text format.")

print(f"Loaded {len(documents):,} document sections from {DATA_PATH}.")

The splitter is intentionally transparent and corpus-specific. Production ingestion should preserve stable document IDs and structured metadata rather than infer titles with a regular expression.

## 2. Create overlapping chunks

In [ ]:
CHUNK_SIZE = 900
OVERLAP = 200


def chunk_text(work, text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append(
            {
                "work": work,
                "start": start,
                "end": end,
                "text": text[start:end],
            }
        )
        if end == len(text):
            break
    return chunks


all_chunks = [
    {**chunk, "document_id": document_id}
    for document_id, document in enumerate(documents)
    for chunk in chunk_text(document["work"], document["text"])
]

print(f"Created {len(all_chunks):,} chunks.")
print(f"Example chunk length: {len(all_chunks[0]['text']):,} characters")

## 3. Embed the chunks and build a FAISS index

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMBEDDING_MODEL)

embeddings = encoder.encode(
    [chunk["text"] for chunk in all_chunks],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Indexed {index.ntotal:,} vectors with {embeddings.shape[1]} dimensions.")

Because the vectors are normalized, inner product is equivalent to cosine similarity. Higher retrieval scores indicate greater embedding similarity, not proof that a passage answers the question.

## 4. Retrieve candidate evidence

In [ ]:
def retrieve(query, k=5):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a nonempty string")
    if not isinstance(k, int) or isinstance(k, bool) or k <= 0:
        raise ValueError("k must be a positive integer")

    k = min(k, len(all_chunks))
    query_embedding = encoder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores, indices = index.search(query_embedding, k)
    return [
        {
            "rank": rank,
            "score": float(score),
            **all_chunks[int(chunk_index)],
        }
        for rank, (score, chunk_index) in enumerate(
            zip(scores[0], indices[0]),
            start=1,
        )
    ]


def preview_hits(hits, excerpt_length=180):
    for hit in hits:
        excerpt = " ".join(hit["text"][:excerpt_length].split())
        print(
            f"[{hit['rank']}] {hit['work']} "
            f"(score={hit['score']:.3f}, chars={hit['start']}-{hit['end']})"
        )
        print(f"    {excerpt}…")


sample_hits = retrieve("What does Hamlet say about being and not being?", k=5)
preview_hits(sample_hits)

## 5. Build a grounded prompt

In [ ]:
def build_prompt(question, hits):
    context = "\n\n---\n\n".join(
        f"[{hit['rank']}] {hit['work']} "
        f"(chars {hit['start']}-{hit['end']}):\n{hit['text']}"
        for hit in hits
    )
    system = (
        "Answer only from the supplied context. "
        "Cite supporting passages with bracketed numbers such as [1] or [2]. "
        "If the context does not support an answer, say exactly: "
        "'I don't know based on the provided context.' "
        "Do not use outside knowledge."
    )
    user = f"Question: {question}\n\nContext:\n{context}\n\nAnswer:"
    return system, user

## 6. Generate an answer locally with Ollama

In [ ]:
import ollama


def ask_llm(system, user, model_name="llama3:8b"):
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        stream=False,
    )
    return response["message"]["content"]


def answer_question(question, k=5, model_name="llama3:8b"):
    hits = retrieve(question, k=k)
    system, user = build_prompt(question, hits)
    answer = ask_llm(system, user, model_name=model_name)
    return answer, hits

## 7. Test abstention on an out-of-domain question

Shakespeare's works cannot support an answer about a Harry Potter character. The desired behavior is an explicit abstention. Inspect both the retrieved passages and the generated answer—RAG does not guarantee grounding by itself.

In [ ]:
question = "How did Dumbledore die?"
answer, hits = answer_question(question)

preview_hits(hits)
print("\nModel answer:\n")
print(answer)

## Checks

In [ ]:
assert len(documents) > 0
assert len(all_chunks) == index.ntotal
assert all(0 < len(chunk["text"]) <= CHUNK_SIZE for chunk in all_chunks)
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4)

for document_id, document in enumerate(documents):
    document_chunks = [
        chunk for chunk in all_chunks if chunk["document_id"] == document_id
    ]
    assert document_chunks[0]["start"] == 0
    assert document_chunks[-1]["end"] == len(document["text"])
    assert all(
        chunk["text"] == document["text"][chunk["start"] : chunk["end"]]
        for chunk in document_chunks
    )
    assert all(
        right["start"] == left["end"] - OVERLAP
        for left, right in zip(document_chunks, document_chunks[1:])
    )

print("Corpus, chunk coverage, index, and embedding checks passed.")

## Limitations and next steps

This is an educational prototype, not a production RAG service. In particular:

- the corpus parser is heuristic and citations use character offsets;
- retrieval quality is not evaluated with labeled questions and relevance judgments;
- an abstention instruction is not a guarantee against hallucination;
- retrieved text is not isolated against prompt injection;
- indexes are rebuilt in memory and there is no deployment hardening.

Useful next steps are a small evaluation set with Recall@k and answer-grounding checks, document-level metadata, persisted indexes, configurable models, and adversarial prompt-injection tests.